# Exercises

In this section we have two exercises:

1. Implement the polynomial kernel.
2. Implement the multiclass C-SVM.


## Polynomial kernel

You need to extend the `build_kernel` function and implement the polynomial kernel if the `kernel_type` is set to 'poly'. The equation that needs to be implemented:
\begin{equation}
K=(X^{T}*Y)^{d}.
\end{equation}


In [13]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

iris = load_iris()
X, y = iris.data, iris.target

train_data_set, test_data_set, train_labels, test_labels = train_test_split(
    X, y, test_size=0.2, random_state=42
)

data_set = (train_data_set, test_data_set, train_labels, test_labels)

In [14]:
import numpy as np


def build_kernel(data_set, kernel_type="linear"):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == "rbf":
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (
            np.dot((np.diag(kernel) * np.ones((1, objects_count))).T, b.T)
            + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T)
        )
        kernel = np.exp(kernel / (2.0 * sigma**2))
    elif kernel_type == "poly":
        # put your code here
        d = 2
        kernel = kernel**d
    return kernel

## Implement a multiclass C-SVM

Use the classification method that we used in notebook 7.3 and IRIS dataset to build a multiclass C-SVM classifier. Most implementation is about a function that will return the proper data set that need to be used for the prediction. You need to implement:

- `choose_set_for_label`
- `get_labels_count`


In [15]:
def choose_set_for_label(data_set, label):
    # your code here
    train_data_set, test_data_set, train_labels, test_labels = data_set
    _train_labels = np.where(train_labels == label, 1, -1)
    _test_labels = np.where(test_labels == label, 1, -1)
    return train_data_set, test_data_set, _train_labels, _test_labels

In [16]:
def get_labels_count(data_set):
    _, _, labels, _ = data_set
    labels_count = len(np.unique(labels))
    return labels_count

Use the code that we have implemented earlier:


In [17]:
import cvxopt


def train(train_data_set, train_labels, kernel_type="linear", C=10, threshold=1e-5):
    objects_count = len(train_data_set)
    kernel = build_kernel(train_data_set, kernel_type=kernel_type)

    P = train_labels * train_labels.transpose() * kernel
    q = -np.ones((objects_count, 1))
    G = np.concatenate((np.eye(objects_count), -np.eye(objects_count)))
    h = np.concatenate((C * np.ones((objects_count, 1)), np.zeros((objects_count, 1))))

    A = train_labels.reshape(1, objects_count)
    A = A.astype(float)
    b = 0.0

    sol = cvxopt.solvers.qp(
        cvxopt.matrix(P),
        cvxopt.matrix(q),
        cvxopt.matrix(G),
        cvxopt.matrix(h),
        cvxopt.matrix(A),
        cvxopt.matrix(b),
    )

    lambdas = np.array(sol["x"])

    support_vectors_id = np.where(lambdas > threshold)[0]
    vector_number = len(support_vectors_id)
    support_vectors = train_data_set[support_vectors_id, :]

    lambdas = lambdas[support_vectors_id]
    targets = train_labels[support_vectors_id]

    b = np.sum(targets)
    for n in range(vector_number):
        b -= np.sum(
            lambdas
            * targets
            * np.reshape(
                kernel[support_vectors_id[n], support_vectors_id], (vector_number, 1)
            )
        )
    b /= len(lambdas)

    return lambdas, support_vectors, support_vectors_id, b, targets, vector_number


def build_kernel(data_set, kernel_type="linear"):
    kernel = np.dot(data_set, data_set.T)
    if kernel_type == "rbf":
        sigma = 1.0
        objects_count = len(data_set)
        b = np.ones((len(data_set), 1))
        kernel -= 0.5 * (
            np.dot((np.diag(kernel) * np.ones((1, objects_count))).T, b.T)
            + np.dot(b, (np.diag(kernel) * np.ones((1, objects_count))).T.T)
        )
        kernel = np.exp(kernel / (2.0 * sigma**2))
    return kernel


def classify_rbf(
    test_data_set,
    train_data_set,
    lambdas,
    targets,
    b,
    vector_number,
    support_vectors,
    support_vectors_id,
):
    kernel = np.dot(test_data_set, support_vectors.T)
    sigma = 1.0
    # K = np.dot(test_data_set, support_vectors.T)
    # kernel = build_kernel(train_data_set, kernel_type='rbf')
    c = (
        1.0
        / sigma
        * np.sum(test_data_set**2, axis=1)
        * np.ones((1, np.shape(test_data_set)[0]))
    ).T
    c = np.dot(c, np.ones((1, np.shape(kernel)[1])))
    # aa = np.dot((np.diag(K)*np.ones((1,len(test_data_set)))).T[support_vectors_id], np.ones((1, np.shape(K)[0]))).T
    sv = (
        np.diag(np.dot(train_data_set, train_data_set.T))
        * np.ones((1, len(train_data_set)))
    ).T[support_vectors_id]
    aa = np.dot(sv, np.ones((1, np.shape(kernel)[0]))).T
    kernel = kernel - 0.5 * c - 0.5 * aa
    kernel = np.exp(kernel / (2.0 * sigma**2))

    y = np.zeros((np.shape(test_data_set)[0], 1))
    for j in range(np.shape(test_data_set)[0]):
        for i in range(vector_number):
            y[j] += lambdas[i] * targets[i] * kernel[j, i]
        y[j] += b
    return y

In [18]:
# modify this part
from sklearn.metrics import accuracy_score

unique_labels = np.unique(train_labels)
labels_count = get_labels_count(data_set)
predicted = np.zeros((len(test_data_set), labels_count))

for idx, label in enumerate(unique_labels):
    cur_train_data, cur_test_data, cur_train_labels, cur_test_labels = (
        choose_set_for_label(data_set, label)
    )
    lambdas, support_vectors, support_vectors_id, b, targets, vector_number = train(
        cur_train_data, cur_train_labels, kernel_type="rbf"
    )
    predicted[:, idx] = classify_rbf(
        cur_test_data,
        cur_train_data,
        lambdas,
        targets,
        b,
        vector_number,
        support_vectors,
        support_vectors_id,
    ).flatten()

predicted = unique_labels[np.argmax(predicted, axis=1)]
print(accuracy_score(predicted, test_labels))

     pcost       dcost       gap    pres   dres
 0:  1.1634e+02 -1.8056e+03  4e+03  2e-01  2e-15
 1:  7.7529e+01 -1.6880e+02  3e+02  5e-03  3e-15
 2:  1.0115e+01 -2.2189e+01  3e+01  2e-15  3e-15
 3: -4.8661e-01 -4.9596e+00  4e+00  8e-16  1e-15
 4: -1.4222e+00 -2.4756e+00  1e+00  2e-16  5e-16
 5: -1.6628e+00 -2.1247e+00  5e-01  2e-16  2e-16
 6: -1.7800e+00 -1.9873e+00  2e-01  3e-16  3e-16
 7: -1.8258e+00 -1.9518e+00  1e-01  2e-16  3e-16
 8: -1.8609e+00 -1.8731e+00  1e-02  3e-16  3e-16
 9: -1.8660e+00 -1.8662e+00  2e-04  6e-16  3e-16
10: -1.8661e+00 -1.8661e+00  2e-06  2e-16  4e-16
Optimal solution found.
     pcost       dcost       gap    pres   dres
 0:  1.1473e+02 -2.4495e+03  5e+03  2e-01  2e-15
 1:  9.3901e+01 -2.2202e+02  4e+02  6e-03  3e-15
 2:  1.2918e+01 -2.5931e+01  4e+01  2e-05  3e-15
 3: -1.1759e-01 -5.5532e+00  5e+00  7e-16  2e-15
 4: -1.3443e+00 -2.3367e+00  1e+00  4e-16  5e-16
 5: -1.5524e+00 -1.9857e+00  4e-01  2e-16  3e-16
 6: -1.6978e+00 -1.8728e+00  2e-01  2e-16  3e-1